# 01 · Data audit

**What this notebook decides.** Whether the prepared dataset is fit to train and
evaluate on — before any GPU time is spent on it.

Four things are checked, in descending order of how badly they would corrupt a
result:

1. **Split integrity.** Does any database appear on both sides of the
   train/validation split? This is the failure that would invalidate every
   downstream number, and it is silent.
2. **Record integrity.** Duplicates, missing fields, unreachable databases.
3. **Prompt budget.** Are the largest schemas being silently truncated?
4. **Composition.** How concentrated is the data across databases and difficulty?

In [ ]:
from pathlib import Path

import pandas as pd

from qwen_text2sql.reporting import (
    artifact,
    dataset_fingerprint,
    environment_report,
    preflight,
    project_root,
    schema_statistics,
)
from qwen_text2sql.reporting.style import use_house_style

ROOT = project_root()
PALETTE = use_house_style("light")
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")
ROOT

In [ ]:
report = environment_report()
print(f"commit   {report['git_commit']}")
print(f"python   {report['python']}")
print(f"platform {report['platform']}")
pd.Series(report["packages"], name="version").to_frame()

## Prerequisites

Missing inputs are reported here rather than surfacing as an empty table three
cells later.

In [ ]:
pd.DataFrame(preflight("bird_train_all", "bird_train", "bird_validation"))

In [ ]:
from qwen_text2sql.io import read_jsonl

prepared_path = artifact("bird_train_all").require()
examples = pd.DataFrame(list(read_jsonl(prepared_path)))
pd.Series(dataset_fingerprint(prepared_path)).to_frame("value")

## 1 · Split integrity

The single most important check in this notebook. BIRD questions cluster by
database: if one schema appears in both partitions, the model can memorise that
schema during training and be rewarded for it at validation, and every reported
gain becomes partly leakage.

In [ ]:
train = pd.DataFrame(list(read_jsonl(artifact("bird_train").require())))
validation = pd.DataFrame(list(read_jsonl(artifact("bird_validation").require())))

train_dbs = set(train["db_id"])
validation_dbs = set(validation["db_id"])
overlap = train_dbs & validation_dbs

split_summary = pd.Series(
    {
        "train examples": len(train),
        "validation examples": len(validation),
        "train databases": len(train_dbs),
        "validation databases": len(validation_dbs),
        "databases in BOTH": len(overlap),
        "validation share of examples": len(validation) / (len(train) + len(validation)),
    },
    name="value",
).to_frame()

if overlap:
    raise AssertionError(
        f"Split leakage: {len(overlap)} database(s) appear in both partitions: "
        f"{sorted(overlap)[:10]}"
    )
split_summary

The cell above **raises** on leakage rather than printing a warning. A warning
in a long notebook is a warning nobody reads.

## 2 · Record integrity

In [ ]:
required = ["example_id", "db_id", "question", "gold_sql", "schema", "db_path"]
missing_columns = [column for column in required if column not in examples]
if missing_columns:
    raise AssertionError(f"Prepared file is missing columns: {missing_columns}")

integrity = pd.Series(
    {
        "rows": len(examples),
        "databases": examples["db_id"].nunique(),
        "duplicate example_id": int(examples["example_id"].duplicated().sum()),
        "duplicate (db_id, question)": int(examples.duplicated(subset=["db_id", "question"]).sum()),
        "blank question": int(examples["question"].fillna("").str.strip().eq("").sum()),
        "blank gold_sql": int(examples["gold_sql"].fillna("").str.strip().eq("").sum()),
        "blank schema": int(examples["schema"].fillna("").str.strip().eq("").sum()),
    },
    name="count",
).to_frame()
integrity

In [ ]:
sample = examples.sample(min(200, len(examples)), random_state=42)
unreachable = [path for path in sample["db_path"] if not Path(path).is_file()]
print(
    f"Sampled {len(sample)} db_path values; {len(unreachable)} unreachable.\n"
    "db_path is environment-local by design (see docs/data_contract.md); "
    "unreachable paths mean the databases were not extracted where the "
    "preparation step recorded them."
)

## 3 · Prompt budget

Schema text dominates the prompt. If the largest schemas exceed the context
window, those examples are silently truncated — and any accuracy drop on exactly
the databases with the biggest schemas is then an artefact of the prompt, not a
property of the model.

Characters are used as a cheap, tokenizer-free proxy; roughly 3.5–4 characters
per token for SQL and English.

In [ ]:
from qwen_text2sql.config import load_config

config = load_config(ROOT / "configs/qwen35_4b_lora.yaml")
CHARS_PER_TOKEN = 3.6
limit_chars = int(config.model.max_seq_length * CHARS_PER_TOKEN)

stats = schema_statistics(examples.to_dict("records"))
stats[["question_chars", "evidence_chars", "schema_chars", "prompt_chars"]].describe().T

In [ ]:
from qwen_text2sql.reporting import figures

ax = figures.prompt_budget(stats, limit_chars=limit_chars)
ax.figure.suptitle("")
ax

In [ ]:
over_budget = stats[stats["prompt_chars"] > limit_chars]
if over_budget.empty:
    print(f"No example exceeds the ~{limit_chars:,} character budget.")
else:
    print(f"{len(over_budget)} example(s) over budget, concentrated in these databases:")
    display(over_budget["db_id"].value_counts().head(10).to_frame("examples over budget"))

## 4 · Composition

A dataset where a handful of databases supply most examples will produce an
aggregate accuracy that mostly describes those databases.

In [ ]:
per_db = (
    examples.groupby("db_id")
    .agg(
        examples=("example_id", "size"),
        median_schema_chars=("schema", lambda column: column.str.len().median()),
    )
    .sort_values("examples", ascending=False)
)
concentration = pd.Series(
    {
        "databases": len(per_db),
        "median examples per database": per_db["examples"].median(),
        "share of examples in top 5 databases": per_db["examples"].head(5).sum() / len(examples),
        "share of examples in top 20 databases": per_db["examples"].head(20).sum() / len(examples),
        "databases with < 10 examples": int((per_db["examples"] < 10).sum()),
    },
    name="value",
).to_frame()
concentration

In [ ]:
if "difficulty" in examples:
    difficulty = examples["difficulty"].fillna("").replace("", "unlabelled").value_counts()
    display(difficulty.to_frame("examples").assign(share=lambda f: f["examples"] / len(examples)))
else:
    print("No difficulty labels in this release.")

## Reading this audit

**What it shows.** That the split is database-disjoint, that records are
structurally sound, whether any example is truncated by the context window, and
how concentrated the data is.

**What it does not show.** Nothing here validates that a `gold_sql` actually
answers its question. BIRD contains annotation errors, and an example whose
reference query is wrong will be scored as a model failure for the rest of this
sequence. Rule 7 exists for that reason: suspected annotation errors are
recorded separately in [`06_error_analysis`](06_error_analysis.ipynb), never
folded into the model's error rate.

---

### Provenance

Every number above is produced by code in `src/qwen_text2sql/`, which is unit
tested and type checked. This notebook orchestrates and interprets; it does not
define the analysis. If a figure here looks wrong, the fix belongs in `src/` and
comes with a test.